In [1]:
import torch

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from model_v2_compatible import SeqNN

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

/tmp/SLURM_1074787/ipykernel_1314740/2744654124.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [5]:
import pandas as pd

In [6]:
df = pd.read_csv("/scratch1/smaruj/genomic_insertion_loci/fold1_selected_genomic_windows_centered.tsv", sep="\t")

In [7]:
fountain_mask_path = "/scratch1/smaruj/generate_genomic_fountains/fountain_indices.pt"

In [10]:
import sys
sys.path.insert(0, "/home1/smaruj/ledidi")
from ledidi import Ledidi

In [11]:
bin_size = 2048
cropping_applied = 64
padding_bins = 2
padding = padding_bins * bin_size

# 256 +- 25 (50 bins in the center)
slice_0_bins = [i for i in range(231, 281)]
slice_0_start = (min(slice_0_bins) + cropping_applied - padding_bins) * bin_size
slice_0_end = (max(slice_0_bins) + 1 + cropping_applied + padding_bins) * bin_size

In [12]:
for row in df[:2].itertuples(index=False):
    chrom, pred_start, pred_end = row.chrom, row.centered_start, row.centered_end
    
    print(f"Fountain generation for genome location: {chrom}:{pred_start}-{pred_end}")
    
    X = torch.load(f"/scratch1/smaruj/genomic_insertion_loci/ohe_X/fold1/{chrom}_{pred_start}_{pred_end}_X.pt", weights_only=True, map_location=device)
    target = torch.load(f"/scratch1/smaruj/generate_genomic_fountains/targets_0.5/fold1/{chrom}_{pred_start}_{pred_end}_target.pt", weights_only=True, map_location=device)
    tower_output_path = f"/scratch1/smaruj/genomic_insertion_loci/tower_outputs/fold1/{chrom}_{pred_start}_{pred_end}_tower_out.pt"
    
    wrapper = Ledidi(model, 
                 input_loss=torch.nn.L1Loss(reduction='sum'), 
                 output_loss=torch.nn.L1Loss(reduction='sum'),
                 batch_size=1,
                 max_iter=3000,
                 early_stopping_iter=2000,
                 return_history=False,
                 verbose=True,
                 bin_size=2048,
                 input_mask_slices_0=[i for i in range(231, 281)], # mid-bin
                 cropping_applied=64,
                 output_mask_path=fountain_mask_path,
                 use_semifreddo=True,
                 semifreddo_temp_output_path=tower_output_path,
                 punish_ctcf=False,
                 ctcf_meme_path=None
                 ).cuda()
    
    slice_0_torch = X[:, :, slice_0_start:slice_0_end]
    
    x_bar_slice_0, last_update, _, _, _ = wrapper.fit_transform(X=slice_0_torch, y_bar=target)
    
    # torch.save(x_bar_slice_0[:,:,padding:-padding], f"/scratch1/smaruj/genomic_insertion_loci/results/{chrom}_{pred_start}_{pred_end}_slice.pt")
    

Fountain generation for genome location: chr1:50487296-51798016
Model in train mode: False
Gradients enabled for weights - slice 0: True
Weights shape - slice 0: torch.Size([1, 4, 102400])
Local loss applied.
iter=I	input_loss=0.0	output_loss=6.471e+04	total_loss=6.471e+04	time=0.0
iter=100	input_loss=1.868e+04	output_loss=1.511e+04	total_loss=1.698e+04	time=8.232
iter=200	input_loss=1.299e+04	output_loss=1.415e+04	total_loss=1.545e+04	time=8.135
iter=300	input_loss=1.296e+04	output_loss=1.412e+04	total_loss=1.542e+04	time=8.152
iter=400	input_loss=1.364e+04	output_loss=1.398e+04	total_loss=1.534e+04	time=8.15
iter=500	input_loss=1.442e+04	output_loss=1.35e+04	total_loss=1.494e+04	time=8.154
iter=600	input_loss=1.511e+04	output_loss=1.356e+04	total_loss=1.507e+04	time=8.145
iter=700	input_loss=1.575e+04	output_loss=1.322e+04	total_loss=1.48e+04	time=8.16
iter=800	input_loss=1.637e+04	output_loss=1.342e+04	total_loss=1.505e+04	time=8.147
iter=900	input_loss=1.663e+04	output_loss=1.334e+

KeyboardInterrupt: 